In [ ]:
# [0] Colab setup — clone repo so src/ is available, install CLIP
import sys, os

REPO = 'https://github.com/sudikshyapant/Sparse-CLIP-with-Spectral-Loss'
REPO_DIR = '/content/Sparse-CLIP-with-Spectral-Loss'

if 'google.colab' in sys.modules:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO} {REPO_DIR}')
    os.chdir(REPO_DIR)
    os.system('pip install -q git+https://github.com/openai/CLIP.git')

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Working dir:', repo_root)

# Variation 1 — InfoNCE vs Spectral Contrastive Loss

In [ ]:
# [1] Config — mounts Drive, sets all paths
from google.colab import drive
drive.mount('/content/drive')

from src.config import CONFIG
print('cache_dir :', CONFIG['cache_dir'])
print('device    :', CONFIG['device'])

In [ ]:
# [2] Download COCO images to local storage (skipped if cache already on Drive)
#
# First run:  downloads images locally → computes embeddings → saves .pt to Drive
# Later runs: cache found on Drive → this cell is a no-op

import os, pathlib

DRIVE_COCO = '/content/drive/MyDrive/sparse_clip/coco'
LOCAL_COCO = '/content/coco'
cache_dir  = CONFIG['cache_dir']

need_train = not (cache_dir / 'train_img_emb.pt').exists()
need_val   = not (cache_dir / 'val_img_emb.pt').exists()

if not need_train and not need_val:
    print('Cache found on Drive — skipping all downloads.')
else:
    os.makedirs(f'{LOCAL_COCO}/annotations', exist_ok=True)

    # Annotations — try Drive first, fall back to direct download
    ann_needed = []
    for f in ['captions_train2017.json', 'captions_val2017.json']:
        dst = f'{LOCAL_COCO}/annotations/{f}'
        if not os.path.exists(dst):
            drive_src = f'{DRIVE_COCO}/annotations/{f}'
            if os.path.exists(drive_src):
                os.system(f'cp {drive_src} {dst}')
                print(f'Copied {f}')
            else:
                ann_needed.append(f)
    if ann_needed:
        print('Downloading annotations (~240 MB)...')
        os.system('wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip')
        os.system(f'unzip -jo /tmp/ann.zip "annotations/captions_*.json" -d {LOCAL_COCO}/annotations')
        os.system('rm /tmp/ann.zip')
        print('Annotations ready')

    if need_train:
        print('Downloading train2017 (~18 GB) to local storage...')
        os.system('wget -q http://images.cocodataset.org/zips/train2017.zip -O /tmp/train2017.zip')
        os.system(f'unzip -q /tmp/train2017.zip -d {LOCAL_COCO}')
        os.system('rm /tmp/train2017.zip')
        print('train2017 ready')

    if need_val:
        if os.path.exists(f'{DRIVE_COCO}/val2017'):
            print('Copying val2017 from Drive...')
            os.system(f'cp -r {DRIVE_COCO}/val2017 {LOCAL_COCO}/val2017')
        else:
            print('Downloading val2017 (~1 GB)...')
            os.system('wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip')
            os.system(f'unzip -q /tmp/val2017.zip -d {LOCAL_COCO}')
            os.system('rm /tmp/val2017.zip')
        print('val2017 ready')

    # Point CONFIG to local images for fast reading
    CONFIG['coco_train_images'] = pathlib.Path(LOCAL_COCO) / 'train2017'
    CONFIG['coco_val_images']   = pathlib.Path(LOCAL_COCO) / 'val2017'
    CONFIG['coco_train_ann']    = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_train2017.json'
    CONFIG['coco_val_ann']      = pathlib.Path(LOCAL_COCO) / 'annotations' / 'captions_val2017.json'
    print('CONFIG paths updated to local storage')

In [ ]:
# [2b] Verify annotations — re-download only if copy from Drive failed
import os

LOCAL_COCO = '/content/coco'
missing = [f for f in ['captions_train2017.json', 'captions_val2017.json']
           if not os.path.exists(f'{LOCAL_COCO}/annotations/{f}')]

if missing:
    print(f'Missing: {missing} — downloading annotations (~240 MB)...')
    os.system('wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/ann.zip')
    os.system(f'unzip -jo /tmp/ann.zip "annotations/captions_*.json" -d {LOCAL_COCO}/annotations')
    os.system('rm /tmp/ann.zip')
    print('Annotations ready')
else:
    print('Annotations OK')

In [ ]:
# [3] Compute and cache CLIP embeddings
# Reads images locally (first run) or loads .pt from Drive (subsequent runs).
import clip, torch
from src.data_utils import cache_or_compute_embeddings, make_loader

device = CONFIG['device']
clip_model, preprocess = clip.load(CONFIG['clip_model'], device=device)
clip_model.eval()

train_img, train_txt = cache_or_compute_embeddings(clip_model, preprocess, 'train', CONFIG)
val_img,   val_txt   = cache_or_compute_embeddings(clip_model, preprocess, 'val',   CONFIG)
print(f'train: {train_img.shape}  val: {val_img.shape}')

BATCH_SIZE = 512
train_loader = make_loader(train_img, train_txt, BATCH_SIZE)

In [ ]:
# [4] Model factory
from src.model import SparseHead

def make_head():
    return SparseHead(CONFIG['embed_dim'], CONFIG['sparse_dim']).to(device)

In [ ]:
# [5a] Train — InfoNCE with learnable logit scale
import torch.nn as nn, torch.optim as optim
from src.losses import infonce_loss
from src.train  import evaluate, save_checkpoint, make_run_tag

EPOCHS      = 45
ACCUM_STEPS = 16   # effective batch = 512 × 16 = 8192
run_tag     = make_run_tag(EPOCHS, BATCH_SIZE * ACCUM_STEPS)
print(f'Run tag: {run_tag}')

head_infonce = make_head()
log_scale = nn.Parameter(torch.tensor(CONFIG['log_scale_init'], device=device))

def infonce_fn(img, txt):
    return infonce_loss(img, txt, log_scale)

opt_infonce = optim.AdamW(
    list(head_infonce.parameters()) + [log_scale],
    lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)

loss_curve_infonce = []
for epoch in range(1, EPOCHS + 1):
    head_infonce.train()
    total_loss = 0.0
    opt_infonce.zero_grad()
    for step, (img_emb, txt_emb) in enumerate(train_loader):
        img_emb, txt_emb = img_emb.to(device), txt_emb.to(device)
        _, img_z = head_infonce(img_emb)
        _, txt_z = head_infonce(txt_emb)
        loss = infonce_fn(img_z, txt_z) / ACCUM_STEPS
        loss.backward()
        total_loss += loss.item()
        if (step + 1) % ACCUM_STEPS == 0:
            opt_infonce.step()
            opt_infonce.zero_grad()
    epoch_loss = total_loss / len(train_loader)
    loss_curve_infonce.append(epoch_loss)
    print(f'InfoNCE  epoch {epoch:3d}/{EPOCHS}  loss={epoch_loss:.4f}  τ={1/log_scale.exp().item():.4f}')

metrics_infonce = evaluate(head_infonce, val_img, val_txt, CONFIG)
metrics_infonce['loss_curve'] = loss_curve_infonce
print('\nInfoNCE metrics:', metrics_infonce)
save_checkpoint(head_infonce, metrics_infonce, 'infonce', 'variation1', CONFIG, run_tag)

In [ ]:
# [5b] Train — Spectral contrastive loss
import torch.optim as optim
from src.losses import spectral_loss

EPOCHS      = 45
ACCUM_STEPS = 16   # effective batch = 512 × 16 = 8192
run_tag     = make_run_tag(EPOCHS, BATCH_SIZE * ACCUM_STEPS)  # same tag as 5a

head_spectral = make_head()
opt_spectral = optim.AdamW(
    head_spectral.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay']
)

loss_curve_spectral = []
for epoch in range(1, EPOCHS + 1):
    head_spectral.train()
    total_loss = 0.0
    opt_spectral.zero_grad()
    for step, (img_emb, txt_emb) in enumerate(train_loader):
        img_emb, txt_emb = img_emb.to(device), txt_emb.to(device)
        _, img_z = head_spectral(img_emb)
        _, txt_z = head_spectral(txt_emb)
        loss = spectral_loss(img_z, txt_z) / ACCUM_STEPS
        loss.backward()
        total_loss += loss.item()
        if (step + 1) % ACCUM_STEPS == 0:
            opt_spectral.step()
            opt_spectral.zero_grad()
    epoch_loss = total_loss / len(train_loader)
    loss_curve_spectral.append(epoch_loss)
    print(f'Spectral epoch {epoch:3d}/{EPOCHS}  loss={epoch_loss:.4f}')

metrics_spectral = evaluate(head_spectral, val_img, val_txt, CONFIG)
metrics_spectral['loss_curve'] = loss_curve_spectral
print('\nSpectral metrics:', metrics_spectral)
save_checkpoint(head_spectral, metrics_spectral, 'spectral', 'variation1', CONFIG, run_tag)

In [ ]:
# [6] Results table
k = CONFIG['retrieval_k']
print(f'{"Model":12s}  IR@1   TR@1   IR@5   TR@5   L0_img   Active%  Clarity  Cross-modal')
for name, m in [('InfoNCE', metrics_infonce), ('Spectral', metrics_spectral)]:
    print(f"{name:12s}  {m[f'IR@{k}']:.3f}   {m[f'TR@{k}']:.3f}   "
          f"{m.get('IR@5', float('nan')):.3f}   {m.get('TR@5', float('nan')):.3f}   "
          f"{m['l0_img']:6.1f}   {m.get('active_pct', float('nan')):5.1f}%  "
          f"{m['clarity']:.4f}   {m['cross_modal']:.4f}")

In [ ]:
# [7] Comparison plot
import matplotlib.pyplot as plt

names = ['InfoNCE', 'Spectral']
all_m = [metrics_infonce, metrics_spectral]
line_styles = ['-', '--']
k = CONFIG['retrieval_k']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

# — Loss curves (solid = InfoNCE, dashed = Spectral)
for name, m, ls in zip(names, all_m, line_styles):
    axes[0].plot(m['loss_curve'], label=name, linestyle=ls)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

# — Retrieval@K
x = range(len(names))
ir = [m[f'IR@{k}'] for m in all_m]
tr = [m[f'TR@{k}'] for m in all_m]
axes[1].bar([i - 0.2 for i in x], ir, 0.4, label=f'IR@{k}')
axes[1].bar([i + 0.2 for i in x], tr, 0.4, label=f'TR@{k}')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(names)
axes[1].set_title(f'Retrieval@{k}'); axes[1].legend()

# — Clarity vs L0 (twin y-axis)
cl = [m['clarity'] for m in all_m]
l0 = [m['l0_img']  for m in all_m]
ax2 = axes[2].twinx()
axes[2].bar([i - 0.2 for i in x], cl, 0.4, color='steelblue', label='Clarity')
ax2.bar(    [i + 0.2 for i in x], l0, 0.4, color='orange',    label='L0 img')
axes[2].set_xticks(list(x)); axes[2].set_xticklabels(names)
axes[2].set_ylabel('Clarity', color='steelblue')
ax2.set_ylabel('L0', color='orange')
axes[2].set_title('Clarity vs L0')
axes[2].legend(loc='upper left'); ax2.legend(loc='upper right')

# — Cross-modal score (0.5 = perfectly balanced image/text activations)
cm = [m['cross_modal'] for m in all_m]
bars = axes[3].bar(list(x), cm, 0.5, color='mediumseagreen')
axes[3].axhline(0.5, color='red', linestyle='--', linewidth=1, label='ideal (0.5)')
axes[3].set_xticks(list(x)); axes[3].set_xticklabels(names)
axes[3].set_ylim(0, 1)
axes[3].set_ylabel('Cross-modal score')
axes[3].set_title('Cross-modal balance')
axes[3].legend()
for bar, v in zip(bars, cm):
    axes[3].text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=9)

fig.tight_layout()
out_path = CONFIG['results_dir'] / 'variation1' / f'comparison_{run_tag}.png'
fig.savefig(out_path, dpi=150)
plt.show()
print(f'Saved {out_path}')

In [ ]:
# [8] CIFAR-100 zero-shot classification
# Tests whether sparse representations preserve CLIP's zero-shot transfer ability.
# CIFAR-100 images (32×32 → upsampled to 224) are embedded with frozen CLIP,
# then classified via nearest sparse text prototype.
import torch.nn.functional as F
from src.data_utils import load_or_compute_cifar100_embs
from src.metrics    import zero_shot_accuracy

cifar_img, cifar_cls, cifar_labels = load_or_compute_cifar100_embs(
    clip_model, preprocess, CONFIG['cache_dir'], device
)

# Baseline: raw CLIP cosine similarity (no SparseHead)
with torch.no_grad():
    raw_sims = cifar_img.to(device) @ cifar_cls.to(device).T
raw_acc = (raw_sims.argmax(1).cpu() == cifar_labels).float().mean().item()

print(f'\nCIFAR-100 Zero-Shot Accuracy (Acc@1)')
print(f'{"Model":14s}  Acc@1   vs raw CLIP')
print(f'{"CLIP (raw)":14s}  {raw_acc:.3f}   baseline')
for name, head_model in [('InfoNCE', head_infonce), ('Spectral', head_spectral)]:
    acc = zero_shot_accuracy(head_model, cifar_cls, cifar_img, cifar_labels, device)
    delta = acc - raw_acc
    sign  = '+' if delta >= 0 else ''
    print(f'{name:14s}  {acc:.3f}   {sign}{delta:.3f}')

In [ ]:
# [9] Modality score distribution (Figure 3b in paper)
# Shows what fraction of features are truly multimodal (ratio ≈ 0.5)
# vs. image-only (ratio ≈ 1) or text-only (ratio ≈ 0).
import matplotlib.pyplot as plt
from src.visualization import plot_modality_grid

modality_data = {}
for name, head_model in [('InfoNCE', head_infonce), ('Spectral', head_spectral)]:
    head_model.eval()
    with torch.no_grad():
        z_img, _ = head_model(val_img.to(device))
        z_txt, _ = head_model(val_txt.to(device))
    modality_data[name] = (z_img.cpu(), z_txt.cpu())

fig = plot_modality_grid(modality_data, ncols=2,
                         out_path=CONFIG['results_dir'] / 'variation1' / f'modality_{run_tag}.png')
plt.show()
print('Saved modality distribution plot.')

In [ ]:
# [10] Feature activation heatmap + retrieval similarity matrix
# Heatmap: rows = top-active features, cols = val samples — shows how features
#          fire selectively across the dataset.
# Similarity matrix: cosine sim between sparse image and text reps — the
#          diagonal should be bright (positive pairs matched).
from src.visualization import plot_eval_grid

eval_data = {}
for name, head_model in [('InfoNCE', head_infonce), ('Spectral', head_spectral)]:
    head_model.eval()
    with torch.no_grad():
        z_img, img_z = head_model(val_img.to(device))
        z_txt, txt_z = head_model(val_txt.to(device))
    eval_data[name] = {
        'z_img': z_img.cpu(), 'z_txt': z_txt.cpu(),
        'img_z': img_z.cpu(), 'txt_z': txt_z.cpu(),
    }

fig = plot_eval_grid(eval_data,
                     out_path=CONFIG['results_dir'] / 'variation1' / f'eval_grid_{run_tag}.png')
plt.show()
print('Saved evaluation grid (modality / heatmap / similarity matrix).')